In [ ]:
# =========================
# Cell 1. 导入库
# =========================

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from rasterio.transform import xy as r_xy
# =========================
# Cell 2. 读取训练样本
# =========================

df = pd.read_csv(
    config.SAMPLE_CA,
    sep=r"\s+",
    header=None
)

# 只取前三列
df = df.iloc[:, :3]

df.columns = ["x", "y", "label"]

df["x"] = df["x"].astype(float)
df["y"] = df["y"].astype(float)
df["label"] = df["label"].astype(int)

# 正样本
pos_df = df[df["label"] == 1].copy()

print("=" * 60)
print("正样本数量：", len(pos_df))
print("=" * 60)

# =========================
# Cell 3. 读取栅格
# =========================

features = {
    "Pb": config.RAW_GEOCHEM["Pb"],
    "Zn": config.RAW_GEOCHEM["Zn"],
    "Cu": config.RAW_GEOCHEM["Cu"],
    "As": config.RAW_GEOCHEM["As"],
    "Au": config.RAW_GEOCHEM["Au"],
    "Sb": config.RAW_GEOCHEM["Sb"],
    "Hg": config.RAW_GEOCHEM["Hg"],
    "Ag": config.RAW_GEOCHEM["Ag"],
    "Ip-ρ": config.RAW_IP_RHO,
    "Ip-M": config.RAW_IP_M,
    "Magnetic": config.RAW_MAG,
}

# 参考栅格
ref = rasterio.open(list(features.values())[0])

transform = ref.transform

print("=" * 60)
print("读取栅格完成")
print("因子数量：", len(features))
print("栅格大小：", ref.width, "×", ref.height)
print("=" * 60)

# =========================
# Cell 4. 栅格采样函数
# =========================

def sample_raster(path, points):

    with rasterio.open(path) as src:

        data = src.read(1)

        vals = []

        for x, y in points:

            row, col = src.index(x, y)

            if 0 <= row < data.shape[0] and 0 <= col < data.shape[1]:

                vals.append(data[row, col])

            else:

                vals.append(np.nan)

    return np.array(vals)

# =========================
# Cell 5. 构建固定负样本
# =========================

print("="*60)
print("开始构建固定负样本")
print("="*60)

ag = rasterio.open(features["Ag"]).read(1)
pb = rasterio.open(features["Pb"]).read(1)
zn = rasterio.open(features["Zn"]).read(1)

ag_th = np.nanpercentile(ag,30)
pb_th = np.nanpercentile(pb,30)
zn_th = np.nanpercentile(zn,30)

print("Ag 30% =",ag_th)
print("Pb 30% =",pb_th)
print("Zn 30% =",zn_th)

low_mask = (
    (ag<ag_th) &
    (pb<pb_th) &
    (zn<zn_th)
)

rows_neg, cols_neg = np.where(low_mask)

print("候选负样本数量：",len(rows_neg))

np.random.seed(42)

neg_num = len(pos_df)*2

neg_num = min(neg_num,len(rows_neg))

choose = np.random.choice(
    len(rows_neg),
    neg_num,
    replace=False
)

neg_points = []

for i in choose:

    x,y = ref.xy(rows_neg[i],cols_neg[i])

    neg_points.append((x,y))

print("固定负样本数量：",len(neg_points))

# =========================
# Cell 6. 提取训练样本特征
# =========================

print("="*60)
print("开始提取训练样本特征")
print("="*60)

# 正样本
pos_points = pos_df[["x","y"]].values

X_pos = []

for name,path in features.items():

    vals = sample_raster(path,pos_points)

    X_pos.append(vals)

X_pos = np.array(X_pos).T

y_pos = np.ones(len(X_pos))

# --------------------------
# 负样本
# --------------------------

X_neg = []

for name,path in features.items():

    vals = sample_raster(path,neg_points)

    X_neg.append(vals)

X_neg = np.array(X_neg).T

y_neg = np.zeros(len(X_neg))

# --------------------------
# 合并
# --------------------------

X = np.vstack([X_pos,X_neg])

y = np.hstack([y_pos,y_neg])

mask = ~np.isnan(X).any(axis=1)

X = X[mask]

y = y[mask]

print("最终训练集：",X.shape)
print("正样本：",np.sum(y==1))
print("负样本：",np.sum(y==0))
print("="*60)




# =========================
# Cell 7. LOOCV
# （固定负样本，仅留出正样本）
# =========================

print("="*60)
print("开始LOOCV")
print("="*60)

positive_index = np.where(y == 1)[0]

prob = np.zeros(len(positive_index))

for i, test_pos in enumerate(positive_index):

    train_positive = np.delete(positive_index, i)

    negative_index = np.where(y == 0)[0]

    train_index = np.concatenate([
        train_positive,
        negative_index
    ])

    X_train = X[train_index]
    y_train = y[train_index]

    X_test = X[test_pos].reshape(1, -1)

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    prob[i] = rf.predict_proba(X_test)[0, 1]

    print(f"{i+1}/{len(positive_index)} 完成")

print("="*60)
print("LOOCV完成")
print("="*60)

# =========================
# Cell 8. Rank
# =========================

rank_percent = []

print("="*60)
print("LOOCV Rank")
print("="*60)

for i, p in enumerate(prob):

    rank = np.sum(prob >= p) / len(prob) * 100

    rank_percent.append(rank)

    print(
        f"矿点{i+1:02d} "
        f"Probability={p:.4f} "
        f"Rank={rank:.2f}%"
    )

rank_percent = np.array(rank_percent)

# =========================
# Cell 9. Top20%
# =========================

TopK = 20

hit = rank_percent <= TopK

print()
print("="*60)
print("Top20%结果")
print("="*60)

for i in range(len(rank_percent)):

    if hit[i]:
        flag = "√命中"
    else:
        flag = "×未命中"

    print(
        f"矿点{i+1:02d} "
        f"Rank={rank_percent[i]:6.2f}% "
        f"{flag}"
    )

accuracy = np.mean(hit)

print()
print(f"Top20% 命中率 = {accuracy*100:.2f}%")

In [ ]:
# =========================
# Cell 9. 使用全部样本训练最终RF
# =========================

from sklearn.ensemble import RandomForestClassifier

print("=" * 60)
print("开始训练最终RF")
print("=" * 60)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

print("最终RF训练完成")

train_prob = rf.predict_proba(X)[:,1]

print("训练集概率范围：")
print(train_prob.min())
print(train_prob.max())

# =========================
# Cell 10. 读取全部栅格（自动对齐）
# =========================

# 参考栅格
with rasterio.open(list(features.values())[0]) as ref:

    ref_shape = (ref.height, ref.width)

    rows = ref.height

    cols = ref.width

    profile = ref.profile.copy()
    
    transform = ref.transform
# -------------------------
# 自动对齐函数
# -------------------------

def align_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1).astype(np.float32)

        out = np.full(
            ref_shape,
            np.nan,
            dtype=np.float32
        )

        hh = min(rows, data.shape[0])

        ww = min(cols, data.shape[1])

        out[:hh, :ww] = data[:hh, :ww]

    return out

# -------------------------
# 读取所有栅格
# -------------------------

raster_arrays_onlyrf = []

for name, path in features.items():

    raster_arrays_onlyrf.append(
        align_raster(path)
    )

raster_arrays_onlyrf = np.stack(
    raster_arrays_onlyrf,
    axis=0
)

print("栅格数组形状：")

print(raster_arrays_onlyrf.shape)
# =========================
# Cell 11. 构建全区预测矩阵
# =========================

n_features, rows, cols = raster_arrays_onlyrf.shape

X_all_onlyrf = raster_arrays_onlyrf.reshape(
    n_features,
    rows * cols
).T

print("展开后：")
print(X_all_onlyrf.shape)

# -------------------------
# 去除无效值
# -------------------------

X_clean_onlyrf = X_all_onlyrf.copy()

X_clean_onlyrf[
    X_clean_onlyrf > 1e30
] = np.nan

X_clean_onlyrf[
    X_clean_onlyrf < -1e30
] = np.nan

X_clean_onlyrf[
    np.isinf(X_clean_onlyrf)
] = np.nan

valid_mask_onlyrf = (
    ~np.isnan(X_clean_onlyrf).any(axis=1)
)

for i in range(X_clean_onlyrf.shape[1]):

    feature = X_clean_onlyrf[:,i]

    p1 = np.nanpercentile(feature,1)

    p99 = np.nanpercentile(feature,99)

    valid_mask_onlyrf &= (
        (feature>=p1) &
        (feature<=p99)
    )

X_valid_onlyrf = X_clean_onlyrf[
    valid_mask_onlyrf
]

print("有效像元：")
print(X_valid_onlyrf.shape)

# =========================
# Cell 12. 全区预测
# =========================

proba_valid_onlyrf = rf.predict_proba(
    X_valid_onlyrf
)[:,1]

print("预测完成")

print(
    proba_valid_onlyrf.min(),
    proba_valid_onlyrf.max()
)

proba_map_onlyrf = np.full(
    rows*cols,
    np.nan
)

proba_map_onlyrf[
    valid_mask_onlyrf
] = proba_valid_onlyrf

proba_map_onlyrf = proba_map_onlyrf.reshape(
    rows,
    cols
)

print("概率图：")

print(proba_map_onlyrf.shape)

# =========================
# Cell 13. 导出GeoTIFF
# =========================

print("=" * 60)
print("开始导出GeoTIFF")
print("=" * 60)

profile = ref.profile.copy()

profile.update(
    dtype="float32",
    count=1,
    nodata=-9999,
    compress="lzw"
)

out_dir = config.ensure_dir(config.OUTPUT / "rf_baseline")
output_tif = str(out_dir / "RF_result.tif")

with rasterio.open(
    output_tif,
    "w",
    **profile
) as dst:

    out = proba_map_onlyrf.copy()

    out[np.isnan(out)] = -9999

    dst.write(
        out.astype(np.float32),
        1
    )

print("GeoTIFF导出完成")

print(output_tif)

# =========================
# Cell 14. 导出TXT
# =========================

print("=" * 60)
print("开始导出TXT")
print("=" * 60)

output_txt = str(out_dir / "RF_prediction.txt")

coords = []

for row in range(rows):

    for col in range(cols):

        prob = proba_map_onlyrf[row, col]

        if np.isnan(prob):

            continue

         x, y = r_xy(transform, row, col)

        coords.append([
            x,
            y,
            prob
        ])

coords = np.array(coords)

np.savetxt(
    output_txt,
    coords,
    fmt="%.6f",
    header="X Y Probability",
    comments=""
)

print("TXT导出完成")

print(output_txt)

# =========================
# Cell 15. Prediction-Area Curve
# =========================

import matplotlib.pyplot as plt

print("=" * 60)
print("开始计算P-A曲线")
print("=" * 60)

# --------------------------
# 全区有效概率
# --------------------------

prob_all = proba_map_onlyrf.flatten()

prob_all = prob_all[
    ~np.isnan(prob_all)
]

# --------------------------
# 九个矿点概率
# --------------------------

mine_prob = []

for idx in positive_index:

    mine_prob.append(
        rf.predict_proba(
            X[idx].reshape(1,-1)
        )[0,1]
    )

mine_prob = np.array(
    mine_prob
)

print("矿点数量：",len(mine_prob))

# --------------------------
# 百分位阈值
# --------------------------

area_percent = []

deposit_percent = []

percentiles = np.arange(
    100,
    0,
    -1
)

for p in percentiles:

    threshold = np.percentile(
        prob_all,
        p
    )

    area = np.sum(
        prob_all >= threshold
    )

    area = area / len(prob_all)

    deposit = np.sum(
        mine_prob >= threshold
    )

    deposit = deposit / len(mine_prob)

    area_percent.append(
        area*100
    )

    deposit_percent.append(
        deposit*100
    )

area_percent = np.array(
    area_percent
)

deposit_percent = np.array(
    deposit_percent
)

# --------------------------
# 绘图
# --------------------------

plt.figure(figsize=(6,6))

plt.plot(
    area_percent,
    deposit_percent,
    lw=2,
    label="RF"
)

plt.xlabel(
    "Predicted Area (%)",
    fontsize=13
)

plt.ylabel(
    "Deposits Captured (%)",
    fontsize=13
)

plt.xlim(0,100)

plt.ylim(0,100)

plt.grid(True)

plt.legend()

plt.tight_layout()

plt.show()

print("P-A曲线完成")